# Replicating the fund-manager trade-prediction paper

One holdings parquet is all that is needed. `Run All` executes every configuration;
each one calls `R.free()` afterwards so five configs will not exhaust memory.

## What is being computed

The paper labels its tables with roman numerals. Below they are named for what they
actually do, and each section spells out the computation.

| paper's name | name used here | question it answers |
|---|---|---|
| Table X | **fund sort** | Do funds whose trades are harder to predict earn more afterwards? |
| Table XII | **stock sort (stock return)** | Do stocks whose trades are harder to predict earn more? |
| — | **stock sort (holders' return)** | Are hard-to-predict stocks held by funds that do better? *(extension)* |

## Step 0 — the prediction target

For each (fund, security, quarter) the label is the direction of the **next quarter's**
share change, with a ±1% dead band:

$$\Delta sh_{i,s,t}=\frac{shares_{i,s,t+1}-shares_{i,s,t}}{|shares_{i,s,t}|+1},\qquad
Y_{i,s,t}=\begin{cases}-1 & \Delta sh \le -1\%\ \ \text{(sell)}\\
0 & |\Delta sh| < 1\%\ \ \text{(hold)}\\ +1 & \Delta sh \ge +1\%\ \ \text{(buy)}\end{cases}$$

A rolling window trains on 20 quarters and predicts the next 8, stepping forward 8 at a
time, so every prediction is out-of-sample.

$$\text{correct}_{i,s,t}=\mathbb{1}\!\left[\hat{Y}_{i,s,t}=Y_{i,s,t}\right]$$

Everything downstream is an aggregation of this one indicator.

## Three timing conventions

`accuracy(t)` asks "did we predict the t → t+1 trade right?", which requires seeing
`shares[t+1]` — knowable only at **t+1**, and public only after the ~45–60 day filing delay,
i.e. partway through **t+2**. So the return window matters:

| | sort variable × return window | status |
|---|---|---|
| `contemporaneous` | accuracy(t) × return t → t+1 | **overlaps its own measurement window; biased** |
| `predictive` | accuracy(t) × return t+1 → t+2 | no overlap; ignores the filing delay ← **default** |
| `tradeable` | accuracy(t) × return t+2 → t+3 | also clears the delay |

All three are always computed. `BASE.eval_timing` only picks which one the summary lines print.

## Two holding conventions (fund-level results only)

| | meaning |
|---|---|
| **actual** | each quarter uses that quarter's reported holdings → includes the effect of later trading |
| **frozen** | weights locked at t, each stock compounds on its own → measures only the portfolio held at t |

The difference between them is the contribution of subsequent rebalancing.

## t-statistics

CRET over h quarters uses **overlapping** windows (t and t+1 share h−1 quarters), so the
quarterly series is autocorrelated and a plain OLS t is overstated. Reported `t{h}` are
**Newey-West** with `lags = h-1`; `t{h}_ols` is shown alongside for comparison.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from dataclasses import replace
%load_ext autoreload
%autoreload 2
import company_replication as R
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

# Confirms which FILE is actually imported and that it has the expected Config fields.
R.check_version()

## Configuration

Point `data_path` at your parquet; if column names differ, edit `col_map`
(keys are the names in *your* file).

Built version-tolerantly: keys this copy of `company_replication.py` does not know about
are dropped with a warning rather than raising `unexpected keyword argument`, and the
absolute path of the imported module is printed so a path mismatch is obvious.

In [ ]:
CFG_KW = dict(
    data_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    eval_timing = "predictive",   # which timing the summary lines print
    inv_type_codes = (401,),
    max_rank   = 25,              # keep top-N positions; also the N used by panel_lstm
    template_N = 75,              # template width used only for the padding calculation
    min_years  = 7, min_holdings = 10,
    window_q   = 28, test_q = 8, step = 8,
    drop_missing_position = True, # chg_pct == -100% means the position info is missing
    enforce_sell_feasibility = True,
    feasibility_mode = "loose",   # "loose" | "strict" | "none"
    feasible_only = False,
    use_manager_memory = False,
    model = "gbm",
    seq_len = 8, hidden = 64, dropout = 0.25, lr = 3e-3,
    max_epochs = 25, patience = 5, batch = 8192, device = "auto",
    lstm_max_train = None, lstm_max_rows = None,
)

import os
known   = set(R.Config.__dataclass_fields__)
dropped = {k: v for k, v in CFG_KW.items() if k not in known}
BASE    = R.Config(**{k: v for k, v in CFG_KW.items() if k in known})

print("module file :", os.path.abspath(R.__file__))
print("version     :", getattr(R, "__version__", "<older copy>"))
print("!! not supported by this copy:", list(dropped)) if dropped else print("all keys accepted")
BASE

## Build the panel (once, shared by every configuration)

Check the printed **class balance**. If the "hold" share is very small, the
`future_1q_shares_change_pct` column follows a different convention than assumed and every
downstream number needs recalibrating.

In [ ]:
panel = R.load_and_prepare(BASE)
print(f"\npanel {len(panel):,} rows | {panel.fund.nunique():,} funds | {panel.qi.max()+1} quarters")
RESULTS = {}

### Volume features — check the units

`pos_to_vol` = `shares / volume`. If `volume` is a dollar amount while `shares` is a share
count the ratio is not interpretable (it may still predict, but it is not a
days-to-liquidate measure). A median near 1e-6 signals that mismatch; switch to
`position_value / volume` in that case. `vol_rank` and `amihud` are unaffected.

In [ ]:
volf = [c for c in ("log_volume", "vol_rank", "pos_to_vol", "d_log_vol", "amihud")
        if c in panel.columns]
if volf:
    display(panel[volf].describe().loc[["count","mean","std","min","25%","50%","75%","max"]].round(4))
    med = panel["pos_to_vol"].median()
    print(f"pos_to_vol median = {med:.6g}")
    print("-> units look consistent" if med > 1e-3 else
          "-> WARNING: very small; volume may be dollars. Consider position_value / volume")
else:
    print("no volume column -- volume features skipped")
print(f"\nfeatures used: {len([f for f in BASE.features if f in panel.columns])}")

## How each result is computed

### 1. Prediction accuracy, with and without padded slots

Accuracy on the real positions is just the mean of `correct`. The paper additionally lays
each fund-quarter onto a fixed **template of N slots**; a fund holding fewer than N names
leaves the rest empty (*padding*). A padded slot holds nothing, so its share change is 0,
its label is "hold" by construction, and it is trivially predicted. If those slots are
scored, accuracy is lifted mechanically:

$$\text{precision}_{\text{with padding}} = p + (1-p)\cdot\text{accuracy}_{\text{real}},
\qquad p=\text{padded share of slots}$$

The table reports several template widths so the sensitivity is visible.

### 2. Fund sort

Per fund-quarter, the sorting variable and the return:

$$\text{precision}_{i,t}=\frac{1}{|S_{i,t}|}\sum_{s} \text{correct}_{i,s,t},
\qquad r^{\text{fund}}_{i,t}=\frac{\sum_s w_{s}(t)\, r_s(t\!\to\! t{+}1)}{\sum_s w_{s}(t)}$$

Abnormal return removes the cross-sectional mean of that quarter; CRET cumulates h quarters
starting at the offset the timing convention dictates (0 / 1 / 2):

$$\text{abn}_{i,t}=r^{\text{fund}}_{i,t}-\overline{r^{\text{fund}}_{\cdot,t}},
\qquad \text{CRET}_{0,h}(i,t)=\sum_{j=0}^{h-1}\text{abn}_{i,\,t+\text{start}+j}$$

Funds are then split into quintiles on `precision`, and Q5−Q1 is the spread.
**frozen** replaces the cumulated abnormal returns with a buy-and-hold compound from t:

$$\text{CRET}^{\text{frozen}}_{0,h}(i,t)=\sum_s w_s(t)\prod_{j=0}^{h-1}\bigl(1+r_s(t{+}\text{start}{+}j)\bigr)-1$$

If any quarter in the window is missing for that fund the value is NaN — never summed
across a gap.

### 3. Stock sort

Per stock-quarter, average `correct` across **all funds holding it**:

$$\text{acc}_{s,t}=\frac{1}{|F_{s,t}|}\sum_{i \in F_{s,t}}\text{correct}_{i,s,t}$$

Stocks are split into quintiles on this, and scored two ways:

- **the stock's own return** — the paper's version
- **its holders' returns** — weight-average of the CRET of every fund holding it *(extension)*

$$\text{holder\_ret}(s,t)=\frac{\sum_{i\in F_{s,t}} w_{i,s,t}\,\text{CRET}_{0,4}(i,t)}{\sum_{i\in F_{s,t}} w_{i,s,t}}$$

Note the stock sort has no actual/frozen distinction for the stock's own return — rebalancing
is a fund-level concept.

In [ ]:
CONFIGS = {
    "A_gbm_no_mem":   dict(model="gbm",        use_manager_memory=False),
    "B_gbm_mem":      dict(model="gbm",        use_manager_memory=True),
    "C_lstm_no_mem":  dict(model="lstm",       use_manager_memory=False),
    "D_lstm_mem":     dict(model="lstm",       use_manager_memory=True),
    "E_panel_no_mem": dict(model="panel_lstm", use_manager_memory=False),
}
CONFIGS

---
# A — gradient boosting, no manager memory

Fastest configuration; run it first to confirm the data conventions are right.

In [ ]:
RESULTS["A_gbm_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["A_gbm_no_mem"]), "A_gbm_no_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["A_gbm_no_mem"]

print("### 1. prediction accuracy (real positions vs incl. padded slots)")
display(r["precision_table"].round(4))

print("### 2. FUND sort: funds ranked by predictability -> their future returns")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- actual rebalancing / {tm} ---"); display(r["fund_sort_actual"][tm].round(3))
    print(f"--- frozen buy-and-hold / {tm} ---"); display(r["fund_sort_frozen"][tm].round(3))

print("### 3. STOCK sort: stocks ranked by cross-fund accuracy")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- the STOCK's own return / {tm} ---");      display(r["stock_sort_stock_ret"][tm].round(3))
    print(f"--- its HOLDERS' returns, actual / {tm} ---"); display(r["stock_sort_holder_ret_actual"][tm].round(3))
    print(f"--- its HOLDERS' returns, frozen / {tm} ---"); display(r["stock_sort_holder_ret_frozen"][tm].round(3))

---
# B — gradient boosting, **with** manager memory

Adds expanding fund / fund-security historical trade rates. These change what "predictable"
means: a manager who never touches a position becomes trivially predictable. Whether that
shifts the return sorts is exactly what this comparison against A measures.

In [ ]:
RESULTS["B_gbm_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["B_gbm_mem"]), "B_gbm_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["B_gbm_mem"]

print("### 1. prediction accuracy (real positions vs incl. padded slots)")
display(r["precision_table"].round(4))

print("### 2. FUND sort: funds ranked by predictability -> their future returns")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- actual rebalancing / {tm} ---"); display(r["fund_sort_actual"][tm].round(3))
    print(f"--- frozen buy-and-hold / {tm} ---"); display(r["fund_sort_frozen"][tm].round(3))

print("### 3. STOCK sort: stocks ranked by cross-fund accuracy")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- the STOCK's own return / {tm} ---");      display(r["stock_sort_stock_ret"][tm].round(3))
    print(f"--- its HOLDERS' returns, actual / {tm} ---"); display(r["stock_sort_holder_ret_actual"][tm].round(3))
    print(f"--- its HOLDERS' returns, frozen / {tm} ---"); display(r["stock_sort_holder_ret_frozen"][tm].round(3))

---
# C — sequence LSTM, no manager memory

The paper uses an LSTM, so this is the architecture-level replication. One sample is one
position's last 8 quarters, `[T, F]`, with weights shared across positions.
Slower than gradient boosting on CPU; set `lstm_max_train=300_000` if it drags.

In [ ]:
RESULTS["C_lstm_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["C_lstm_no_mem"]), "C_lstm_no_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["C_lstm_no_mem"]

print("### 1. prediction accuracy (real positions vs incl. padded slots)")
display(r["precision_table"].round(4))

print("### 2. FUND sort: funds ranked by predictability -> their future returns")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- actual rebalancing / {tm} ---"); display(r["fund_sort_actual"][tm].round(3))
    print(f"--- frozen buy-and-hold / {tm} ---"); display(r["fund_sort_frozen"][tm].round(3))

print("### 3. STOCK sort: stocks ranked by cross-fund accuracy")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- the STOCK's own return / {tm} ---");      display(r["stock_sort_stock_ret"][tm].round(3))
    print(f"--- its HOLDERS' returns, actual / {tm} ---"); display(r["stock_sort_holder_ret_actual"][tm].round(3))
    print(f"--- its HOLDERS' returns, frozen / {tm} ---"); display(r["stock_sort_holder_ret_frozen"][tm].round(3))

---
# D — sequence LSTM, with manager memory

In [ ]:
RESULTS["D_lstm_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["D_lstm_mem"]), "D_lstm_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["D_lstm_mem"]

print("### 1. prediction accuracy (real positions vs incl. padded slots)")
display(r["precision_table"].round(4))

print("### 2. FUND sort: funds ranked by predictability -> their future returns")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- actual rebalancing / {tm} ---"); display(r["fund_sort_actual"][tm].round(3))
    print(f"--- frozen buy-and-hold / {tm} ---"); display(r["fund_sort_frozen"][tm].round(3))

print("### 3. STOCK sort: stocks ranked by cross-fund accuracy")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- the STOCK's own return / {tm} ---");      display(r["stock_sort_stock_ret"][tm].round(3))
    print(f"--- its HOLDERS' returns, actual / {tm} ---"); display(r["stock_sort_holder_ret_actual"][tm].round(3))
    print(f"--- its HOLDERS' returns, frozen / {tm} ---"); display(r["stock_sort_holder_ret_frozen"][tm].round(3))

---
# E — panel LSTM, the paper's own `(T, N, F) → (N × 3)` architecture

One sample is a fund-quarter's **entire cross-section**: column j is the security ranked
j-th in that fund at t, tracked back through time. Funds holding fewer than N names leave
the surplus columns as **padding**.

This configuration prints the **padding share** — direct evidence of how much of the
headline accuracy comes from empty slots. Slowest of the five; reduce `max_rank` if memory
is tight.

In [ ]:
RESULTS["E_panel_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["E_panel_no_mem"]), "E_panel_no_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["E_panel_no_mem"]

print("### 1. prediction accuracy (real positions vs incl. padded slots)")
display(r["precision_table"].round(4))

print("### 2. FUND sort: funds ranked by predictability -> their future returns")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- actual rebalancing / {tm} ---"); display(r["fund_sort_actual"][tm].round(3))
    print(f"--- frozen buy-and-hold / {tm} ---"); display(r["fund_sort_frozen"][tm].round(3))

print("### 3. STOCK sort: stocks ranked by cross-fund accuracy")
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"--- the STOCK's own return / {tm} ---");      display(r["stock_sort_stock_ret"][tm].round(3))
    print(f"--- its HOLDERS' returns, actual / {tm} ---"); display(r["stock_sort_holder_ret_actual"][tm].round(3))
    print(f"--- its HOLDERS' returns, frozen / {tm} ---"); display(r["stock_sort_holder_ret_frozen"][tm].round(3))

---
# Summary across configurations

`fund_sign_matches_paper = yes` means that configuration reproduced the direction of the
paper's fund-level result under the headline timing (less predictable funds outperform).

In [ ]:
summary = R.summarize(RESULTS)
summary

In [ ]:
hz = BASE.eval_timing
cmp = pd.DataFrame([
    {"metric": "accuracy (real positions)", "paper": np.nan,
     **{k: round(v["precision"]["accuracy"], 4) for k, v in RESULTS.items()}},
    {"metric": f"precision (incl. padding, N={BASE.template_N})", "paper": 0.71,
     **{k: round(v["precision_table"].iloc[2]["precision"], 4) for k, v in RESULTS.items()}},
    {"metric": f"naive (incl. padding, N={BASE.template_N})", "paper": 0.52,
     **{k: round(v["precision_table"].iloc[2]["naive"], 4) for k, v in RESULTS.items()}},
    {"metric": f"FUND sort Q5-Q1 CRET_0_4 ({hz}, actual)", "paper": -0.79,
     **{k: round(v["fund_sort_actual"][hz].iloc[-1].CRET_0_4, 3) for k, v in RESULTS.items()}},
    {"metric": f"FUND sort Q5-Q1 CRET_0_4 ({hz}, frozen)", "paper": np.nan,
     **{k: round(v["fund_sort_frozen"][hz].iloc[-1].CRET_0_4, 3) for k, v in RESULTS.items()}},
    {"metric": "STOCK sort Q1-Q5, stock return (contemporaneous)", "paper": 1.06,
     **{k: round(v["stock_sort_stock_ret"]["contemporaneous"].iloc[-1].mean_qret, 3) for k, v in RESULTS.items()}},
    {"metric": f"STOCK sort Q1-Q5, holders' return ({hz}, actual)", "paper": np.nan,
     **{k: round(v["stock_sort_holder_ret_actual"][hz].iloc[-1].holder_CRET, 3) for k, v in RESULTS.items()}},
    {"metric": f"STOCK sort Q1-Q5, holders' return ({hz}, frozen)", "paper": np.nan,
     **{k: round(v["stock_sort_holder_ret_frozen"][hz].iloc[-1].holder_CRET, 3) for k, v in RESULTS.items()}},
])
cmp

## Save

In [ ]:
import os
os.makedirs("outputs_company", exist_ok=True)
for tag, r in RESULTS.items():
    r["precision_table"].to_csv(f"outputs_company/accuracy_{tag}.csv", index=False)
    for tm in ("predictive", "tradeable", "contemporaneous"):
        r["fund_sort_actual"][tm].to_csv(f"outputs_company/fund_sort_actual_{tag}_{tm}.csv", index=False)
        r["fund_sort_frozen"][tm].to_csv(f"outputs_company/fund_sort_frozen_{tag}_{tm}.csv", index=False)
        r["stock_sort_stock_ret"][tm].to_csv(f"outputs_company/stock_sort_stockret_{tag}_{tm}.csv", index=False)
        r["stock_sort_holder_ret_actual"][tm].to_csv(f"outputs_company/stock_sort_holderret_actual_{tag}_{tm}.csv", index=False)
        r["stock_sort_holder_ret_frozen"][tm].to_csv(f"outputs_company/stock_sort_holderret_frozen_{tag}_{tm}.csv", index=False)
summary.to_csv("outputs_company/summary.csv", index=False)
cmp.to_csv("outputs_company/compare_with_paper.csv", index=False)
print("saved to outputs_company/")

## If memory runs short

```python
R.free(RESULTS)                                          # keep tables, drop prediction detail

c = replace(BASE, model="panel_lstm", max_rank=15)       # smaller N -> smaller tensor
c = replace(BASE, model="lstm", lstm_max_train=300_000)  # cap train sequences per window
c = replace(BASE, model="lstm", lstm_max_rows=2_000_000) # subsample the panel first
```

Sequences are assembled lazily from indices, so a 10M-row panel costs roughly 2 GB for the
sequence models rather than 7 GB. `panel_lstm` scales with `max_rank`.